# Data Visualization

The purpose of this notebook is to practice data visualization, and hopefully communicate some best-practices along the way.


# Please upvote if you find this useful

Some sources I'd reccomend for data visualization principles are:

- Storytelling with data, C. Knaflic

- The visual display of quantitative information, E. Tufte

- Better data visualizations, J. Schwabish


I have other notebooks that also incoroprate some nice visuals:

**UK COVID-19 Vaccination Data Visualization**

https://www.kaggle.com/joshuaswords/uk-covid-19-vaccination-progress-data-vis


**Exploratory Data Visualization - Student Performance**

https://www.kaggle.com/joshuaswords/data-visualisation-student-results

**HR Data Set - Visuals & Predictions**

https://www.kaggle.com/joshuaswords/awesome-hr-data-visualization-prediction


**Visuals & Customer Segmentation**

https://www.kaggle.com/joshuaswords/data-visualization-clustering-mall-data


As well as learning from other Kagglers who enjoy data viz.

With that said, let's go...

In [1]:
import numpy as np
import os

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    import pandas as pd
import os

for dirname, _, filenames in os.walk("../input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

../input/netflix-shows/netflix_titles.csv


In [2]:
import warnings

warnings.filterwarnings("ignore")
import os

In [3]:
df = pd.read_csv(
    "/home/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/input/netflix-shows/netflix_titles.csv"
)
df.head(3)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...


# -- STEFANOS -- Replicate Data

In [4]:
factor = 5
df_cell6 = pd.concat([df] * factor, ignore_index=True)

In [5]:
for i in df_cell6.columns:
    null_rate = df_cell6[i].isna().sum() / len(df_cell6) * 100
    if null_rate > 0:
        print("{} null rate: {}%".format(i, round(null_rate, 2)))

director null rate: 29.91%
cast null rate: 9.37%
country null rate: 9.44%
date_added null rate: 0.11%
rating null rate: 0.05%
duration null rate: 0.03%


- 5 columns have missing values, with Director missing 1/3 of the time

# Dealing with the missing data

- This is always scenario dependant, but in this case, I will:
    - replace blank countries with the mode (most common) country
    - I want to keep director as it could be interesting to look at a certain director's films
    - I want to keep cast as it could be interesting to look at a certain cast's films
    

In [6]:
df_cell6["country"] = df_cell6["country"].fillna(df_cell6["country"].mode()[0])
df_cell6["cast"].replace(np.nan, "No Data", inplace=True)
df_cell6["director"].replace(np.nan, "No Data", inplace=True)
df_cell6.dropna(inplace=True)

In [7]:
df_cell6.isnull().sum()

show_id         0
type            0
title           0
director        0
cast            0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
description     0
dtype: int64

In [8]:
df_cell6.info()

<class 'pandas.core.frame.DataFrame'>
Index: 43950 entries, 0 to 44034
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       43950 non-null  object
 1   type          43950 non-null  object
 2   title         43950 non-null  object
 3   director      43950 non-null  object
 4   cast          43950 non-null  object
 5   country       43950 non-null  object
 6   date_added    43950 non-null  object
 7   release_year  43950 non-null  int64 
 8   rating        43950 non-null  object
 9   duration      43950 non-null  object
 10  listed_in     43950 non-null  object
 11  description   43950 non-null  object
dtypes: int64(1), object(11)
memory usage: 4.4+ MB


# Missing values dealt with, but the date isn't quite right yet...

In [9]:
df_cell6["date_added"] = pd.to_datetime(df_cell6["date_added"], errors="coerce")
df_cell6["month_added"] = df_cell6["date_added"].dt.month
df_cell6["index"] = df_cell6["date_added"].dt.month_name()
df_cell6["year_added"] = df_cell6["date_added"].dt.year
df_cell6.head(3)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,month_added,index,year_added
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,No Data,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",9.0,September,2021.0
1,s2,TV Show,Blood & Water,No Data,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",9.0,September,2021.0
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",United States,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,9.0,September,2021.0


# Okay, let's visualize

# Where possible, I'll use the Netflix brand colours

https://brand.netflix.com/en/assets/brand-symbol/


Using a consistent color palette is a great way to give your work credibility. It looks professional, and keeps the reader engaged. 

It's an easy-to-implement tip that really helps.

# Netflix through the years

Netflix started as DVD rentals, and now they have an audience of over 150m people - this is their story...

Timeline code from Subin An's awesome notebook
https://www.kaggle.com/subinium/awesome-visualization-with-titanic-dataset

# Content - Let's explore

Now we've seen how Netflix came to dominate our TV screens, let's have a look at the content they offer...

In [10]:
x = df_cell6.groupby(["type"])["type"].count()
y = len(df_cell6)
r = (x / y).round(2)
mf_ratio = pd.DataFrame(r).T

In [11]:
for i_1 in mf_ratio.index:
    _ = int(mf_ratio["Movie"][i_1] * 100)
    _ = mf_ratio["Movie"][i_1] / 2
    _ = mf_ratio["Movie"][i_1] / 2
for i_2 in mf_ratio.index:
    _ = int(mf_ratio["TV Show"][i_2] * 100)
    _ = mf_ratio["Movie"][i_2] + mf_ratio["TV Show"][i_2] / 2
    _ = mf_ratio["Movie"][i_2] + mf_ratio["TV Show"][i_2] / 2

# By Country

So we now know there are much more movies than TV shows on Netflix (which surprises me!).

What about if we look at content by country? 

I would imagine that the USA will have the most content. I wonder how my country, the UK, will compare?

In [12]:
df_cell6["count"] = 1
df_cell6["first_country"] = df_cell6["country"].apply(lambda x: x.split(",")[0])
df_cell6["first_country"].head()
ratings_ages = {
    "TV-PG": "Older Kids",
    "TV-MA": "Adults",
    "TV-Y7-FV": "Older Kids",
    "TV-Y7": "Older Kids",
    "TV-14": "Teens",
    "R": "Adults",
    "TV-Y": "Kids",
    "NR": "Adults",
    "PG-13": "Teens",
    "TV-G": "Kids",
    "PG": "Older Kids",
    "G": "Kids",
    "UR": "Adults",
    "NC-17": "Adults",
}
df_cell6["target_ages"] = df_cell6["rating"].replace(ratings_ages)
df_cell6["target_ages"].unique()
df_cell6["genre"] = df_cell6["listed_in"].apply(
    lambda x: x.replace(" ,", ",").replace(", ", ",").split(",")
)
df_cell6["first_country"].replace("United States", "USA", inplace=True)
df_cell6["first_country"].replace("United Kingdom", "UK", inplace=True)
df_cell6["first_country"].replace("South Korea", "S. Korea", inplace=True)

In [14]:
df_cell6.date_added

0       2021-09-25
1       2021-09-24
2       2021-09-24
3       2021-09-24
4       2021-09-24
           ...    
44030   2019-11-20
44031   2019-07-01
44032   2019-11-01
44033   2020-01-11
44034   2019-03-02
Name: date_added, Length: 43950, dtype: datetime64[ns]

In [16]:
data = (
    df_cell6.groupby("first_country")["count"].sum().sort_values(ascending=False)[:10]
)

As predicted, the USA dominates. 

The UK is a top contender too, but still some way behind India.

How does content by country vary? 

In [17]:
country_order = df_cell6["first_country"].value_counts()[:11].index
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = df_cell6._to_pandas()
data_q2q3 = (
    df_cell6[["type", "first_country"]]
    .groupby("first_country")["type"]
    .value_counts()
    .unstack()
    .loc[country_order]
)
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = pd.DataFrame(df_cell6)
    data_q2q3 = pd.DataFrame(data_q2q3)
data_q2q3["sum"] = data_q2q3.sum(axis=1)
data_q2q3_ratio = (
    (data_q2q3.T / data_q2q3["sum"])
    .T[["Movie", "TV Show"]]
    .sort_values(by="Movie", ascending=False)[::-1]
)

As I've noted in the insights on the plot, it is really interesting to see how the split of TV Shows and Movies varies by country.

South Korea is dominated by TV Shows - why is this? I am a huge fan of South Korean cinema so I know they have a great movie selection.

Equally, India is dominated by Movies. I think this might be due to Bollywood - comment below if you have any other ideas!

# Ratings

Let's briefly check out how ratings are distributed

In [18]:
order = pd.DataFrame(
    df_cell6.groupby("rating")["count"].sum().sort_values(ascending=False).reset_index()
)
rating_order = list(order["rating"])

In [19]:
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = df_cell6._to_pandas()
mf = (
    df_cell6.groupby("type")["rating"]
    .value_counts()
    .unstack()
    .sort_index()
    .fillna(0)
    .astype(int)[rating_order]
)
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = pd.DataFrame(df_cell6)
    mf = pd.DataFrame(mf)
movie = mf.loc["Movie"]
tv = -mf.loc["TV Show"]

# How has content been added over the years?

As we saw in the timeline at the start of this analysis, Netflix went global in 2016 - and it is extremely noticeable in this plot.

The increase is Movie content is remarkable.

# We can view the same plot, but as a cumulative total...

In [21]:
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = df_cell6._to_pandas()
data_sub = (
    df_cell6.groupby("type")["year_added"]
    .value_counts()
    .unstack()
    .fillna(0)
    .loc[["TV Show", "Movie"]]
    .cumsum(axis=0)
    .T
)
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = pd.DataFrame(df_cell6)
    data_sub = pd.DataFrame(data_sub)

# Month-by-Month

We've seen how content has increased over the years, but are there certain months that, on average, tend to enjoy more content being added?

I'll show this in a couple of ways - a cumulative year view, and also as a radial plot...

In [22]:
month_order = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December",
]
df_cell6["index"] = pd.Categorical(
    df_cell6["index"], categories=month_order, ordered=True
)

In [23]:
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = df_cell6._to_pandas()
data_sub_cell38 = (
    df_cell6.groupby("type")["index"]
    .value_counts()
    .unstack()
    .fillna(0)
    .loc[["TV Show", "Movie"]]
    .cumsum(axis=0)
    .T
)
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = pd.DataFrame(df_cell6)
    data_sub_cell38 = pd.DataFrame(data_sub_cell38)

# What about a more interesting way to view how content is added across the year?

Sometimes visualizations should be eye-catching & attention grabbing - I think this visual acheives that, even if it isn't the most precise.

By highlighting certain months, the reader's eye is drawn exactly where we want it. 

In [24]:
data_sub2 = data_sub_cell38
data_sub2["Value"] = data_sub2["Movie"] + data_sub2["TV Show"]
data_sub2_cell40 = data_sub2.reset_index()
df_polar = data_sub2_cell40.sort_values(by="index", ascending=False)
color_map = ["#221f1f" for _ in range(12)]
color_map[0] = color_map[11] = "#b20710"
upperLimit = 30
lowerLimit = 1
labelPadding = 30
max = df_polar["Value"].max()
slope = (max - lowerLimit) / max
heights = slope * df_polar.Value + lowerLimit
width = 2 * np.pi / len(df_polar.index)
indexes = list(range(1, len(df_polar.index) + 1))
angles = [element * width for element in indexes]
angles

Index(['index', 'TV Show', 'Movie', 'Value'], dtype='object', name='type')


[0.5235987755982988,
 1.0471975511965976,
 1.5707963267948966,
 2.0943951023931953,
 2.617993877991494,
 3.141592653589793,
 3.665191429188092,
 4.1887902047863905,
 4.71238898038469,
 5.235987755982988,
 5.759586531581287,
 6.283185307179586]

Yes, December & January are definitely the best months for new content. Maybe Netflix knows that people have a lot of time off from work over this period and that it is a good time to reel people in?

February is the worst - why might this be? Ideas welcomed!

# Movie Genres

Let's now explore movie genres a little...

In [25]:
import matplotlib.colors

cmap = matplotlib.colors.LinearSegmentedColormap.from_list(
    "", ["#221f1f", "#b20710", "#f5f5f1"]
)

In [26]:
def genre_heatmap(df, title):
    df["genre"] = df["listed_in"].apply(
        lambda x: x.replace(" ,", ",").replace(", ", ",").split(",")
    )
    Types = []
    for i_3 in df["genre"]:
        Types += i_3
    Types = set(Types)
    print("There are {} types in the Netflix {} Dataset".format(len(Types), title))
    test_1 = df["genre"]


df_tv = df_cell6[df_cell6["type"] == "TV Show"]
df_movies = df_cell6[df_cell6["type"] == "Movie"]
genre_heatmap(df_movies, "Movie")

column_view::get_data: Unsupported type: 24
column_view::get_data: Unsupported type: 24
There are 20 types in the Netflix Movie Dataset


In [27]:
data_cell45 = (
    df_cell6.groupby("first_country")[["count"]]
    .sum()
    .sort_values(by="count", ascending=False)
    .reset_index()[:10]
)
data_cell45 = data_cell45["first_country"]
df_heatmap = df_cell6.loc[df_cell6["first_country"].isin(data_cell45)]

column_view::get_data: Unsupported type: 24


In [28]:
df_heatmap_cell46 = pd.crosstab(
    df_heatmap["first_country"], df_heatmap["target_ages"], normalize="index"
).T

# Target Ages

Does Netflix uniformly target certain demographics? Or does this vary by country?



Very interesting results. 

It is also interesting to note similarities between culturally similar countries - the US & UK are closey aligned with their Netflix target ages, yet vastly different to say, India or Japan!

# Let's have a quick look at the lag between when content is released and when it is added on Netflix

Spain looks to have a lot of new content. Great for them!

In [30]:
df_movies
df_tv
data_cell51 = (
    df_movies.groupby("first_country")[["count"]]
    .sum()
    .sort_values(by="count", ascending=False)
    .reset_index()[:10]
)
data_cell51 = data_cell51["first_country"]
df_loli = df_movies.loc[df_movies["first_country"].isin(data_cell51)]
loli = df_loli.groupby("first_country")[["release_year", "year_added"]].mean().round()
ordered_df = loli.sort_values(by="release_year")
ordered_df_rev = loli.sort_values(by="release_year", ascending=False)

What about TV shows...

In [31]:
data_cell53 = (
    df_tv.groupby("first_country")[["count"]]
    .sum()
    .sort_values(by="count", ascending=False)
    .reset_index()[:10]
)
data_cell53 = data_cell53["first_country"]
df_loli_cell53 = df_tv.loc[df_tv["first_country"].isin(data_cell53)]
loli_cell53 = (
    df_loli_cell53.groupby("first_country")[["release_year", "year_added"]]
    .mean()
    .round()
)
ordered_df_cell53 = loli_cell53.sort_values(by="release_year")
ordered_df_rev_cell53 = loli_cell53.sort_values(by="release_year", ascending=False)

column_view::get_data: Unsupported type: 24


In [32]:
us_ind = df_cell6[
    (df_cell6["first_country"] == "USA") | (df_cell6["first_country"] == "India")
]
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = df_cell6._to_pandas()
data_sub_cell54 = (
    df_cell6.groupby("first_country")["year_added"]
    .value_counts()
    .unstack()
    .fillna(0)
    .loc[["USA", "India"]]
    .cumsum(axis=0)
    .T
)
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = pd.DataFrame(df_cell6)
    data_sub_cell54 = pd.DataFrame(data_sub_cell54)

column_view::get_data: Unsupported type: 24


# USA & India

As the two largest content countries, it might be fun to compare the two

In [34]:
us_ind_cell57 = df_cell6[
    (df_cell6["first_country"] == "USA") | (df_cell6["first_country"] == "India")
]
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = df_cell6._to_pandas()
data_sub_cell57 = (
    df_cell6.groupby("first_country")["year_added"]
    .value_counts()
    .unstack()
    .fillna(0)
    .loc[["USA", "India"]]
    .cumsum(axis=0)
    .T
)
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = pd.DataFrame(df_cell6)
    data_sub_cell57 = pd.DataFrame(data_sub_cell57)
data_sub_cell57.insert(0, "base", np.zeros(len(data_sub_cell57)))
data_sub_cell57 = data_sub.add(-us_ind_cell57["year_added"].value_counts() / 2, axis=0)

column_view::get_data: Unsupported type: 24


So the USA dominates. But is there a plot that can convey this in another way?

# Lastly, we can view a wordcloud to get an overview of Netflix titles


It is interesting to note that many films share the same key words in their titles.



Credit to Dmitry Uarov for figuring this visual out. His notebook is here:

https://www.kaggle.com/dmitryuarov/netflix-eda-with-plotly



In [ ]:
import matplotlib

text = (
    str(list(df_cell6["title"]))
    .replace(",", "")
    .replace("[", "")
    .replace("'", "")
    .replace("]", "")
    .replace(".", "")
)


# Thanks for reading!

# I hope you enjoyed my visuals 

# Please consider upvoting if you did

# Have a great day


View more of my work:

**Exploratory Data Visualization - Student Performance**

https://www.kaggle.com/joshuaswords/data-visualisation-student-results

**Visuals & Modelling**

https://www.kaggle.com/joshuaswords/awesome-hr-data-visualization-prediction

**Visuals & Customer Segmentation**

https://www.kaggle.com/joshuaswords/data-visualization-clustering-mall-data

**March 2021 Tabular Playground Series**

https://www.kaggle.com/joshuaswords/tps-eda-model-march-2020

